# Table 2: Server Memory Footprint

Peak Python heap during `scramble_image_grt`, measured with `tracemalloc`, versus the raw RGB buffer size.

In [1]:
import json
import random
import sys
import tracemalloc
from pathlib import Path

import numpy as np
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src" / "cryptography"))
import grt_server

# ----- Configuration: edit this block to change images, grids, or run counts -----
SCENARIOS = [
    {"name": "SD 480p",   "file": "docs/images/t1_SD_480p.png",   "M": 30, "N": 25},
    {"name": "FHD 1080p", "file": "docs/images/t2_FHD_1080p.png", "M": 40, "N": 40},
    {"name": "HD 720p",   "file": "docs/images/t5_HD_720p.png",   "M": 36, "N": 40},
]
WARMUP, ITERS, SEED = 5, 30, 2026
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def make_keys(M, N):
    """Reproducible valid GRT keys (a, b, c, d, rm, rn)."""
    random.seed(SEED)
    np.random.seed(SEED)
    keys = tuple(int(k) for k in grt_server.generate_valid_grt_keys(M, N))
    ok, msg = grt_server.validate_grt_parameters(M, N, *keys)
    assert ok, msg
    return keys


def load_scenario(sc):
    """Load image, pad (edge replication) to a multiple of the grid, return padded image and original size."""
    img = Image.open(ROOT / sc["file"]).convert("RGB")
    padded, orig_size = grt_server.pad_image_to_grid(img, sc["M"], sc["N"])
    return padded, orig_size


def stats(values):
    arr = np.array(values)
    return {"mean": float(arr.mean()), "std": float(arr.std()), "median": float(np.median(arr))}

In [2]:
rows = []
for sc in SCENARIOS:
    M, N = sc["M"], sc["N"]
    img, orig_size = load_scenario(sc)
    keys = make_keys(M, N)
    raw_kb = img.width * img.height * 3 / 1024.0

    peaks = []
    for _ in range(ITERS):
        tracemalloc.start()
        tracemalloc.reset_peak()
        out = grt_server.scramble_image_grt(img, M, N, *keys)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peaks.append(peak / 1024.0)
    peak_kb = float(np.median(peaks))
    rows.append({
        "scenario": sc["name"], "padded_size": list(img.size), "grid": [M, N],
        "raw_kb": raw_kb, "peak_kb": peak_kb, "ratio": peak_kb / raw_kb,
    })

(RESULTS_DIR / "memory.json").write_text(json.dumps(rows, indent=2))
print(f"{'Scenario':<10} {'Padded':>11} {'Grid':>7} {'Raw (KB)':>12} {'Peak (KB)':>12} {'Ratio':>7}")
for r in rows:
    print(f"{r['scenario']:<10} {r['padded_size'][0]:>5}x{r['padded_size'][1]:<5} {r['grid'][0]:>3}x{r['grid'][1]:<3} "
          f"{r['raw_kb']:>12.1f} {r['peak_kb']:>12.1f} {r['ratio']:>6.2f}x")

Scenario        Padded    Grid     Raw (KB)    Peak (KB)   Ratio
SD 480p      480x875    30x25        1230.5       2463.6   2.00x
FHD 1080p   1920x1080   40x40        6075.0      12162.0   2.00x
HD 720p      720x1280   36x40        2700.0       5405.4   2.00x
